In [1]:
#automatically reload stuff
%load_ext autoreload
%autoreload 2

In [ ]:
# This is the scripts I have to import the data,
# save it as a intermediate json (patient_organ_data)
# then denoise that and save it as a second json (patient_organ_data_denoise)
# final version includes inputed values and the original mask

In [2]:
import os
import Utils
from OrganDataPreprocessing import CamprtOrganData
import Formatting
from Constants import Const
import Cluster
import Models
from glob import glob
from re import findall
import pandas as pd
import numpy as np
import shutil
from pathlib import Path

In [3]:
Const.camprt_dir

'../data/CAMPRT_Centroids/'

In [4]:
from pathlib import Path
import shutil

# oldpath = Path("../data/CAMPRT_Centroids_20260421_old_original").resolve()
# newpath = Path("../data/CAMPRT_Centroids_20260421_new_original").resolve()
# fullpath = Path("../data/CAMPRT_Centroids_20260421_full_original").resolve()
oldpath = Path("../data/CAMPRT_Centroids_20260421_old").resolve()
newpath = Path("../data/CAMPRT_Centroids_20260421_new").resolve()
fullpath = Path("../data/CAMPRT_Centroids_20260421_full").resolve()

old_items = {p.name for p in oldpath.iterdir()}
new_items = {p.name for p in newpath.iterdir()}

duplicates = sorted(old_items & new_items)

print(f"Old items: {len(old_items)}")
print(f"New items: {len(new_items)}")
print(f"Duplicated names: {len(duplicates)}")

print("\nDuplicated items:")
for name in duplicates:
    print(name)

Old items: 427
New items: 319
Duplicated names: 91

Duplicated items:
101417584648
102209659698
102934140275
108136889660
108372343816
111851899440
113809646383
117641907656
121450446153
127149326549
128247687988
130431013577
134670372726
135469220715
137183280417
148385943793
157394176135
157952803487
158046919754
158886568644
160748994446
164841038228
172707438936
173700768546
174749776227
175549909570
179733287115
179922054901
180415564296
185636560244
185661990536
188053898062
191082118003
205171292827
209115582685
212125757397
212126630450
213780491365
224196609722
227353307577
245999091808
252961381072
267472084343
276583502481
277200233633
277663884669
280783117125
285108844062
286774288673
297142600913
304382005273
307821869290
311493344600
317679788976
317705536874
317796938500
320608953458
324426263949
325563256918
328025528354
332371615427
335583988457
337740242307
339350973397
349791419858
351467526814
358675083840
364135542628
395764826099
418603878206
427028983398
4387276

In [5]:
fullpath.mkdir(parents=True, exist_ok=True)

def copy_folder_contents(src: Path, dst: Path, overwrite=True):
    if not src.exists():
        raise FileNotFoundError(f"Source folder does not exist: {src}")
    if not src.is_dir():
        raise NotADirectoryError(f"Source path is not a folder: {src}")

    for item in src.iterdir():
        target = dst / item.name

        if item.is_dir():
            shutil.copytree(
                item,
                target,
                dirs_exist_ok=True
            )
        else:
            if target.exists() and not overwrite:
                continue
            shutil.copy2(item, target)

copy_folder_contents(oldpath, fullpath, overwrite=True)

copy_folder_contents(newpath, fullpath, overwrite=True)

print("Merge completed.")
print(f"Old folder:  {oldpath}")
print(f"New folder:  {newpath}")
print(f"Full folder: {fullpath}")

Merge completed.
Old folder:  /Users/siyuanzhao/Documents/GitHub/QubbedDataAnalysis/data/CAMPRT_Centroids_20260421_old
New folder:  /Users/siyuanzhao/Documents/GitHub/QubbedDataAnalysis/data/CAMPRT_Centroids_20260421_new
Full folder: /Users/siyuanzhao/Documents/GitHub/QubbedDataAnalysis/data/CAMPRT_Centroids_20260421_full


In [6]:
def load_spatial_files(root=None):
    """
    Reads tumor centroid and ROI-tumor distance files for the CAMPRT dataset.
    Each patient has a folder named after the patient id, containing:
        CT_centroid.csv
        CT_distances.csv
    Returns {'id': {'distances': <distfile>, 'doses': <centroid_file>}}
    Only patients with both files are returned.
    """
    root = Const.camprt_dir if root is None else root

    try:
        distance_files = glob(
            os.path.join(root, "**", "*distances.csv"), recursive=True
        )
    except Exception:
        distance_files = []
    try:
        dose_files = glob(os.path.join(root, "**", "*centroid*.csv"), recursive=True)
    except Exception:
        dose_files = []

    def file_id(file):
        # Patient id lives in the parent folder name, not the filename
        # (filenames are fixed: CT_centroid.csv / CT_distances.csv).
        folder = os.path.basename(os.path.dirname(file))
        nums = findall(r"[0-9]+", folder)
        if not nums:
            # Fallback: use the folder name itself as the id (string).
            return folder
        return int(max(nums, key=len))  # take the longest digit run

    dose_dict = {file_id(f): f for f in dose_files}
    dist_dict = {file_id(f): f for f in distance_files}

    dose_ids = set(dose_dict.keys())
    dist_ids = set(dist_dict.keys())
    shared_ids = dose_ids.intersection(dist_ids)

    dropped_ids = dose_ids.symmetric_difference(dist_ids)
    if len(dropped_ids) > 0:
        print("missing doses", dist_ids - shared_ids)
        print("missing distances", dose_ids - shared_ids)

    file_dict = {
        sid: {"doses": dose_dict.get(sid), "distances": dist_dict.get(sid)}
        for sid in shared_ids
    }
    return file_dict


sfiles = load_spatial_files(root=fullpath)
print("")
print(f"total patients loaded: {len(sfiles)}")
for k, v in sfiles.items():
    print("patient id   ", k)
    print("dose file    ", v["doses"])
    print("distance file", v["distances"])
    break


total patients loaded: 655
patient id    271412541441
dose file     /Users/siyuanzhao/Documents/GitHub/QubbedDataAnalysis/data/CAMPRT_Centroids_20260421_full/271412541441/CT_centroid.csv
distance file /Users/siyuanzhao/Documents/GitHub/QubbedDataAnalysis/data/CAMPRT_Centroids_20260421_full/271412541441/CT_distances.csv


In [ ]:
# dose files for this version are structured with coordinates, volume, and min/max/mean dose for each roi
# also not that the GTVXX are tumors, names are inconsistent for these and need cleaning
# GTVn is a nodal (lymph node) tumor, GTVp is a primary tumor
pd.read_csv(sfiles[319933062094]["doses"]).head()

In [ ]:
# Distances are paired ROI inter-region distances.
# This is the minimum distance between the outer edges of the organ
# Note this is also annoying to work with
pd.read_csv(sfiles[319933062094]["distances"]).head()

,Reference ROI,Target ROI,Eucledian Distance (mm),Phi (degrees),Theta (degrees),% of Target Overlap,Eucledian Distance (mm) 5th Percentile
0,Hyoid_bone,Lt_Mastoid,65.47,-39.11,138.66,0.0,68.24
1,Hyoid_bone,Rt_Mastoid,69.37,-42.02,-128.86,0.0,71.44
2,Hyoid_bone,Lt_Brachial_Plexus,30.22,36.23,156.74,0.0,34.08
3,Hyoid_bone,Rt_Brachial_Plexus,32.62,36.30,-147.04,0.0,36.71
4,Hyoid_bone,Brainstem,67.32,-60.08,-177.76,0.0,77.16


In [7]:
# read in the organ info and save it to a default format

# In OrganDataPreprocessing I have a very complicated setup for processing a patient
# which includes a bunch of manual error checkers. See that code for reference, but new Dicom data will have different preprocessing
# In this print out, rename cols gives a list of things I renamed. You can see why I had to implement a generic spellchecking
od = CamprtOrganData()
pdict = od.process_cohort_spatial_dict(sfiles)
pdict.keys()
# Utils.np_dict_to_json(pdict,Const.processed_organ_json)
# del pdict

renamed organs {'Esophagus_U': 'Esophagus', 'Hardpalate': 'Hard_Palate', 'SpinalCord': 'Spinal_Cord'}
renamed organs {'Esophagus_U': 'Esophagus', 'Hardpalate': 'Hard_Palate', 'SpinalCord': 'Spinal_Cord'}
renamed organs {'Esophagus_U': 'Esophagus', 'Hardpalate': 'Hard_Palate', 'SpinalCord': 'Spinal_Cord'}
renamed organs {'Esophagus_U': 'Esophagus', 'Hardpalate': 'Hard_Palate', 'SpinalCord': 'Spinal_Cord'}
renamed organs {'Esophagus_U': 'Esophagus', 'Hardpalate': 'Hard_Palate'}
renamed organs {'Esophagus_U': 'Esophagus', 'Hardpalate': 'Hard_Palate', 'SpinalCord': 'Spinal_Cord'}
renamed organs {'Esophagus_U': 'Esophagus', 'Hardpalate': 'Hard_Palate', 'SpinalCord': 'Spinal_Cord'}
renamed organs {'Esophagus_U': 'Esophagus', 'Hardpalate': 'Hard_Palate', 'SpinalCord': 'Spinal_Cord'}
renamed organs {'Esophagus_U': 'Esophagus', 'Hardpalate': 'Hard_Palate', 'SpinalCord': 'Spinal_Cord'}
renamed organs {'Esophagus_U': 'Esophagus', 'Hardpalate': 'Hard_Palate', 'SpinalCord': 'Spinal_Cord'}
renamed o

dict_keys(['organs', 'patients'])

In [9]:
# so this should check that the values make sense
# currently it implies there is an issue but I don't know what
def get_num_nans_per_organ(sdata):
    # should return a list of the % of patients missing data for each organ
    # for each entry type
    olist = sdata["organs"]
    oarlist = olist + ["gtv"]
    allnan = {}
    for key in ["distances", "volume", "centroids", "mean_dose"]:
        nancount = {o: 0 for o in oarlist}
        arr = Formatting.merged_spatial_array(sdata, key)
        for i, aa in enumerate(arr):
            # find missing organs for individual
            all_nan = np.argwhere(np.isnan(aa).all(axis=1))
            nanset = set([])
            if len(all_nan) < 1:
                continue
            for arg in all_nan[0]:
                nanorgan = oarlist[arg]
                nancount[nanorgan] = nancount[nanorgan] + 1
        nancount = [
            (k, np.round(100 * nancount[k] / arr.shape[0], 4))
            for k, v in nancount.items()
            if v > 0
        ]
        nancount = sorted(nancount, key=lambda x: -x[1])

        allnan[key] = nancount
    return allnan


get_num_nans_per_organ(pdict)

{'distances': [('Spinal_Cord', np.float64(42.2078)),
  ('Esophagus', np.float64(3.4091))],
 'volume': [('Spinal_Cord', np.float64(47.0779)),
  ('Esophagus', np.float64(11.8506)),
  ('gtv', np.float64(1.6234))],
 'centroids': [('Spinal_Cord', np.float64(42.2078)),
  ('Esophagus', np.float64(3.4091))],
 'mean_dose': [('Spinal_Cord', np.float64(47.0779)),
  ('Esophagus', np.float64(11.8506)),
  ('gtv', np.float64(1.6234))]}

In [10]:
# converts the json to a dictionary of arrays with all missing values imputed
# imputation here is the denosing autoencoder (see TSSIM paper), which improves results a bit and fills in missing data
# denoise alpha is the coefficient of how much of the denoising is done
# using the autoencoder used to input missing values [0 = none, 1 = all]
di = Formatting.DataInputer(denoise_alpha=0)
ddict = di.get_formatted_arrays(pdict, retrain=True)

# ddict is a dictionary of format {volume|distances|centroids|mean_dose : np.array}
# arrays are in teh shape n_patients, n_organs, n_channels
# channels are e.g. 3 for centroids (x,y,z), n_organs - 1 (no gtv distances) for distances, and 1 for volume and mean dose
[(k, v.shape) for k, v in ddict.items()]

training stopped on epoch 2777
OrganAutoEncoder(
  (hidden_layers): Sequential(
    (0): Linear(in_features=1845, out_features=200, bias=True)
    (1): ReLU()
    (2): Linear(in_features=200, out_features=100, bias=True)
    (3): ReLU()
    (4): Linear(in_features=100, out_features=20, bias=True)
    (5): Dropout(p=0.2, inplace=False)
    (6): BatchNorm1d(20, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (7): ReLU()
    (8): Linear(in_features=20, out_features=100, bias=True)
    (9): ReLU()
    (10): Linear(in_features=100, out_features=200, bias=True)
    (11): ReLU()
    (12): Linear(in_features=200, out_features=1845, bias=True)
    (13): LeakyReLU(negative_slope=0.1)
  )
  (dropout_layer): Dropout(p=0.5, inplace=False)
  (flatten): Flatten(start_dim=1, end_dim=-1)
)
tensor(0.0508, grad_fn=<MeanBackward0>)
mean_dose
81.1377 0.0499994049991308 142.3262 0.0
81.1377 0.0499994049991308 

distances
258.5 0.0 258.5 0.0
258.5 0.0 

centroids
449.2748707314621 15.0 56

[('mean_dose', (616, 41, 1)),
 ('mean_dose_missing', (616, 41, 1)),
 ('distances', (616, 41, 40)),
 ('distances_missing', (616, 41, 40)),
 ('centroids', (616, 41, 3)),
 ('centroids_missing', (616, 41, 3)),
 ('volume', (616, 41, 1)),
 ('volume_missing', (616, 41, 1))]

In [11]:
# check the error on the neural net for inputing/denoising
# only considers non-missing values
# value are before performing clipping on the inputed values
di.error_report()

,_mean_reconstruction_error (%),mean_denoised,std_denoised,shape_denoised,num_nan_denoised,num_negative_denoised,mean_original,std_original,shape_original,num_nan_original,num_negative_original
key,,,,,,,,,,,
mean_dose,373.577538,31.36246,18.98918,"(616, 41, 1)",0,0,34.49895,21.24741,"(616, 41, 1)",12577,0
distances,11.765503,41.10890,32.44223,"(616, 41, 40)",0,0,41.19286,32.50803,"(616, 41, 40)",449290,0
centroids,5.041700,184.65697,70.07596,"(616, 41, 3)",0,0,184.88189,70.49228,"(616, 41, 3)",28725,0
volume,20.900492,19.95849,31.76873,"(616, 41, 1)",0,0,22.46608,38.43058,"(616, 41, 1)",12577,0


In [12]:
def extract_pdict_gtvs(p):
    gtvlist = []
    for pid, pentry in p["patients"].items():
        plist = []
        for o, oentry in pentry.items():
            if "GTV" in o:
                plist.append((o, oentry))
        gtvlist.append((pid, plist))
    return gtvlist


# turn it into another dictionary again
def inputed_data_dict(pdata, insert_gtvs=True):
    pids = Formatting.get_sorted_pids(pdata)
    organs = pdata["organs"] + ["gtv_composite"]
    main_keys = [k for k in ddict.keys() if "_missing" not in k]
    mask_keys = [k for k in ddict.keys() if k not in main_keys]
    make_dict = lambda keys: {
        int(p): {o: {k: np.nan for k in keys} for o in organs} for p in pids
    }
    inputed_data = make_dict(main_keys)
    data_mask = make_dict(mask_keys)
    for k, varray in ddict.items():
        for pid, v_row in zip(pids, varray):
            for organ, o_col in zip(organs, v_row):
                val = o_col
                if len(val) <= 1:
                    val = val[0]
                if k in mask_keys:
                    data_mask[int(pid)][organ][k.replace("_missing", "")] = val
                else:
                    inputed_data[int(pid)][organ][k] = val
    return_dict = {
        "organs": organs,
        "patient_ids": pids,
        "patients": inputed_data,
        "mask": data_mask,
    }
    if insert_gtvs:
        gtvset = extract_pdict_gtvs(pdata)
        for pid, gtvs in gtvset:
            for k, v in gtvs:
                return_dict["patients"][int(pid)][k] = v
    return return_dict


# I'm going to be honest I have no idea what this does
pdata = inputed_data_dict(pdict)
# [(k,v) for k,v in pdata['patients'][185].items() if 'GTV' in k]

In [ ]:
# saves inputed_ddict
print(Const.denoised_organ_json)
# Utils.np_dict_to_json(pdata, Const.denoised_organ_json,True)

../data/patient_organ_data_denoised.json


In [13]:
def denoised_pdict_to_array(dpd, key, organ_list=None, pids=None):
    patients = dpd["patients"]
    if organ_list is None:
        organ_list = dpd["organs"]
    elif not Utils.iterable(organ_list):
        organ_list = [organ_list]
    if pids is None:
        pids = dpd["patient_ids"]
    array = []
    for pid in pids:
        row = []
        pentry = patients.get(pid)
        for organ in organ_list:
            odata = pentry.get(organ)
            if odata is None:
                row.append(0)
            else:
                oval = odata.get(key)
                row.append(oval)
        array.append(row)
    return np.array(array)


# converts the dictionary to arrays to use for prediction
volumes = denoised_pdict_to_array(pdata, "volume")
distances = denoised_pdict_to_array(pdata, "distances")
denoised_pdict_to_array(pdata, "distances", organ_list=["gtv_composite"])

array([[[ 48.65119483,  23.78144221,  31.0190834 , ...,  34.53931656,
          40.54640639,  29.49608201]],

       [[ 30.40845991,  63.68808026,  87.69866519, ...,  47.81367247,
          57.43214285,  26.73602396]],

       [[ 55.53328344,  23.43080925,  40.09805014, ...,  51.42330962,
          61.49752941,  36.49043305]],

       ...,

       [[ 43.95197307,  30.53507918,  35.32536792, ...,  51.75630046,
          62.97917981,  29.95853652]],

       [[ 41.75783687,  22.9670629 ,  34.60749602, ...,  49.00320006,
          54.29962634,  30.22130617]],

       [[ 27.85882249,  99.26457522, 130.        , ...,  28.96661497,
          34.10373956,  21.66694706]]])

In [14]:
# the end results here is that for the Tssim code, we want a distances in the form n_patients, n_organs + gtv, n_organs
# and volumes of the form n_patients, n_organs + gtv
# where n_organs is defined in Const.organ_list, I think
volumes.shape, distances.shape

((616, 41), (616, 41, 40))

In [15]:
# takes in a dictionary of arrays
# returns a similarity matrix 0-1
# n_jobs > 1 will try to multithread
sim = Models.TssimSimilarity(n_jobs=4)
sim_matrix = sim.get_similarity_matrix(distances, volumes)
sim_matrix

array([[1.        , 0.37229359, 0.41011706, ..., 0.41022567, 0.37144298,
        0.35850089],
       [0.37229359, 1.        , 0.82730687, ..., 0.8309746 , 0.73574677,
        0.7281051 ],
       [0.41011706, 0.82730687, 1.        , ..., 0.36796031, 0.35268693,
        0.30968809],
       ...,
       [0.41022567, 0.8309746 , 0.36796031, ..., 1.        , 0.33090091,
        0.30095867],
       [0.37144298, 0.73574677, 0.35268693, ..., 0.33090091, 1.        ,
        0.34356828],
       [0.35850089, 0.7281051 , 0.30968809, ..., 0.30095867, 0.34356828,
        1.        ]])

In [16]:
# Similarity CLusterer does heirarchical clustering based on a similarity matrix, the method used in the TSSIM paper
# see https://docs.scipy.org/doc/scipy/reference/generated/scipy.cluster.hierarchy.linkage.html for what link does
# see https://docs.scipy.org/doc/scipy/reference/generated/scipy.cluster.hierarchy.fcluster.html for what criterion does
clusters = Cluster.SimilarityClusterer(
    4,
    link="ward",  # linkage method, passed to scipy.cluster.heirarchy.linkage
    criterion="maxclust",  # criterion passed to scipy.cluster.heirarch.fcluster
).fit_predict(sim_matrix)
clusters

array([4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4,
       4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 3, 4, 4, 4, 4,
       4, 3, 4, 4, 4, 4, 4, 4, 4, 3, 4, 4, 4, 4, 4, 4, 4, 3, 4, 4, 4, 3,
       4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 3, 4, 4, 3, 4, 4, 4,
       4, 4, 4, 4, 3, 4, 4, 4, 4, 4, 3, 3, 3, 3, 4, 3, 4, 4, 4, 4, 4, 4,
       4, 3, 4, 4, 4, 3, 4, 3, 4, 4, 4, 3, 4, 3, 4, 3, 4, 4, 4, 4, 4, 3,
       4, 3, 4, 4, 4, 4, 4, 3, 3, 4, 4, 4, 3, 4, 4, 4, 3, 4, 4, 3, 3, 3,
       4, 3, 4, 3, 4, 3, 4, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 4, 3,
       4, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 4, 3, 3, 3, 3, 4, 3, 3, 3, 3, 3,
       3, 3, 4, 3, 3, 3, 3, 3, 3, 3, 3, 1, 3, 3, 4, 3, 3, 3, 3, 3, 3, 3,
       3, 3, 3, 4, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3,
       3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3,
       3, 3, 4, 3, 3, 3, 3, 3, 3, 3, 3, 1, 3, 3, 1, 3, 3, 3, 3, 3, 3, 3,
       3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3,

In [ ]:
pids = pdata["patient_ids"]

result = pd.DataFrame({"patient_id": pids, "cluster": clusters})
print(result)

       patient_id  cluster
0    101923688164        4
1    102209659698        4
2    102231587299        4
3    102700862898        4
4    103512138945        4
..            ...      ...
611  959945632401        2
612  964314877165        2
613  965522448233        2
614  968170678561        1
615  980240390976        2

[616 rows x 2 columns]


In [18]:
result.to_csv("../data/tssim_clusters_20260421_original.csv", index=False)

In [ ]:
# We can also get the most similar patients using the TSSIM similarity matrix
# there are two match_type options, the normal knn (default) and the version in the camprt paper (threshold)
knn = Models.PatientKNN(
    match_type="default",  # default used standard k-nearest neighbors
    default_n_matches=4,  # number of neighbors is match_type = default
)
knn.get_matches(sim_matrix)[0:5]

[array([ 45, 197,  18, 173]),
 array([ 32, 153,  25,  15]),
 array([ 32,  25,  15, 153]),
 array([ 32, 153,  25,  15]),
 array([ 25, 153,  15,  32])]

In [ ]:
knn = Models.PatientKNN(
    match_type="threshold",  # threshold takes a variable number of people
    match_threshold=0.8,  # if similarity is above this value, patient is considered a neighbor
    n_match_bounds=[
        1,
        3,
    ],  # [min,max], lower and upper limit on the number of neighbors to return.
)
knn_matches = knn.get_matches(sim_matrix)
[k for k in knn_matches if len(k) > 1][0:5]

[array([197,  18, 173]),
 array([10,  2]),
 array([ 10,  45, 197]),
 array([10, 15, 45]),
 array([10, 15, 45])]

In [ ]:
# takes patient dictionary object and similarity matrix and returns a dictionary of neighbors, similarity scores, and clusters
# which are the data needed in a camprt style interface
# is similarity matrix is not passed, it computes it using tssim
def predict_patients(
    dpd,
    sim=None,
    n_clusters=4,  # number of clusters to pass to SimilarityClusterer
    return_dict=True,  # whether to return a results dict or just the predictions.
    **knn_args,
):
    if sim is None:
        volumes = denoised_pdict_to_array(pdata, "volume")
        distances = denoised_pdict_to_array(pdata, "distances")
        sim = Models.TssimSimilarity(n_jobs=4).get_similarity_matrix(distances, volumes)
    pids = dpd["patient_ids"]

    clusterer = Cluster.SimilarityClusterer(n_clusters)
    clusters = clusterer.fit_predict(sim)

    knn = Models.PatientKNN(**knn_args)
    knn_matches = knn.get_matches(sim_matrix)

    sim_dict = {}
    for i, pid in enumerate(pids):
        entry = {}
        entry["neighbors"] = [pids[ii] for ii in knn_matches[i]]
        entry["cluster"] = clusters[i]
        entry["similarity"] = sim[i, knn_matches[i]]
        sim_dict[pid] = entry
    if return_dict is False:
        thing = sorted(
            [(k, v["cluster"]) for k, v in sim_dict.items()], key=lambda x: x[0]
        )
        return np.array([t[1] for t in thing])
    return sim_dict


# keyword arguments (besides sim) are passed to the PatientKnn constructure shown above
pp = predict_patients(pdata, sim_matrix, match_type="threshold", match_threshold=0.8)
for i, (patient_id, res) in enumerate(pp.items()):
    print("patient", patient_id)
    print(res)
    if i > 2:
        break

patient 102209659698
{'neighbors': [173700768546], 'cluster': np.int32(1), 'similarity': array([0.62991938])}
patient 102231587299
{'neighbors': [150176613149], 'cluster': np.int32(3), 'similarity': array([0.5242022])}
patient 105852012614
{'neighbors': [150176613149], 'cluster': np.int32(3), 'similarity': array([0.66111352])}
patient 106528681216
{'neighbors': [150176613149], 'cluster': np.int32(3), 'similarity': array([0.65776876])}


In [ ]:
predict_patients(pdata, sim_matrix, match_type="default", return_dict=False)[0:10]

array([1, 3, 3, 3, 3, 3, 3, 3, 3, 3], dtype=int32)